<a href="https://colab.research.google.com/github/futuregunmulju/2026-Signals-and-Systems-Speech-Noise-Reduction/blob/experiment%2Fminju/Challenge1_Speech_Noise_Reduction_Algorithms.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 0. Google Mount

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
cd /content/drive/MyDrive/연세대/신호및시스템/9조_noise_reduction_challenge

/content/drive/MyDrive/연세대/신호및시스템/9조_noise_reduction_challenge


# Data Overview

In [4]:
import pandas as pd
from IPython.display import display

datasets = {
    "train": pd.read_csv("train/samples.csv"),
    "validation": pd.read_csv("validation/samples.csv"),
}

for split, df in datasets.items():
    print(f"{split}: {len(df):,}쌍")
    print("컬럼:", ", ".join(df.columns))
    display(df.head(3))

train: 1,296쌍
컬럼: segment_id, clean_path, noisy_path, sample_rate_hz, num_samples, noise_category, noise_collection, noise_code, noise_difficulty, target_snr_db, split


,segment_id,clean_path,noisy_path,sample_rate_hz,num_samples,noise_category,noise_collection,noise_code,noise_difficulty,target_snr_db,split
0,speaker020_utt01_seg0001,train/clean/speaker020_utt01_seg0001.wav,train/noisy/speaker020_utt01_seg0001_noisy.wav,16000,48000,aircraft_cockpit,NOISEX-92,buccaneer1,easy,10.0,train
1,speaker020_utt01_seg0002,train/clean/speaker020_utt01_seg0002.wav,train/noisy/speaker020_utt01_seg0002_noisy.wav,16000,48000,traffic_vehicle,Nonspeech,N046,moderate,15.0,train
2,speaker020_utt01_seg0003,train/clean/speaker020_utt01_seg0003.wav,train/noisy/speaker020_utt01_seg0003_noisy.wav,16000,48000,alarm_siren,Nonspeech,N036,easy,5.0,train


validation: 164쌍
컬럼: segment_id, clean_path, noisy_path, sample_rate_hz, num_samples, noise_category, noise_collection, noise_code, noise_difficulty, target_snr_db, split


,segment_id,clean_path,noisy_path,sample_rate_hz,num_samples,noise_category,noise_collection,noise_code,noise_difficulty,target_snr_db,split
0,speaker015_utt01_seg0001,validation/clean/speaker015_utt01_seg0001.wav,validation/noisy/speaker015_utt01_seg0001_nois...,16000,48000,aircraft_cockpit,NOISEX-92,f16,easy,10.0,validation
1,speaker015_utt01_seg0002,validation/clean/speaker015_utt01_seg0002.wav,validation/noisy/speaker015_utt01_seg0002_nois...,16000,48000,machine,Nonspeech,N026,moderate,15.0,validation
2,speaker015_utt01_seg0003,validation/clean/speaker015_utt01_seg0003.wav,validation/noisy/speaker015_utt01_seg0003_nois...,16000,48000,alarm_siren,Nonspeech,N035,easy,5.0,validation


In [5]:
all_samples = pd.concat(datasets.values(), ignore_index=True)
for column in ("noise_category", "noise_difficulty", "target_snr_db"):
    counts = pd.crosstab(
        all_samples[column].fillna("Missing"),
        all_samples["split"],
    ).reindex(columns=["train", "validation"], fill_value=0)

    percentages = counts.div(counts.sum(axis=0), axis=1).mul(100).round(1)

    print(f"\n{column}")
    display(pd.concat({"Count": counts, "Percent (%)": percentages}, axis=1))

print("Noise categories across both splits:", all_samples["noise_category"].nunique())


noise_category


Count            Percent (%)           
split                train validation       train validation
noise_category                                              
aircraft_cockpit       111         14         8.6        8.5
alarm_siren            187         23        14.4       14.0
animal                  39          2         3.0        1.2
bell                    15          3         1.2        1.8
broadband_noise         73         10         5.6        6.1
clap                     3          1         0.2        0.6
click                    3          1         0.2        0.6
destroyer_engine        36          5         2.8        3.0
destroyer_operations    36          5         2.8        3.0
door_moving              3          1         0.2        0.6
factory                 38          5         2.9        3.0
footsteps                8          1         0.6        0.6
hf_channel              37          4         2.9        2.4
machine                264         34        20.4       20.7
machine_gun              3          1         0.2        0.6
military_vehicle        37          5         2.9        3.0
phone_dialing           43          4         3.3        2.4
shower                   3          1         0.2        0.6
tooth_brushing           3          1         0.2        0.6
traffic_vehicle         76          9         5.9        5.5
vehicle_interior        19          3         1.5        1.8
water                   57          5         4.4        3.0
wind                   202         26        15.6       15.9


noise_difficulty


Count            Percent (%)           
split            train validation       train validation
noise_difficulty                                        
easy               776         99        59.9       60.4
hard               195         24        15.0       14.6
moderate           325         41        25.1       25.0


target_snr_db


Count            Percent (%)           
split         train validation       train validation
target_snr_db                                        
0.0             117         15         9.0        9.1
5.0             298         38        23.0       23.2
10.0            429         55        33.1       33.5
15.0            452         56        34.9       34.1

Noise categories across both splits: 23


In [9]:
# Install PESQ
%pip install -q pesq==0.0.4

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done


In [11]:
# Load Validation Data
import numpy as np
import pandas as pd
import soundfile as sf

from IPython.display import Audio, display
from pesq import pesq
from scipy.ndimage import convolve1d
from scipy.signal import stft, istft
from scipy.special import exp1, expit

SR = 16_000
validation = pd.read_csv("validation/samples.csv")

def read_validation_pair(row):
    clean, clean_sr = sf.read(row.clean_path, dtype="float64")
    noisy, noisy_sr = sf.read(row.noisy_path, dtype="float64")

    if clean_sr != SR or noisy_sr != SR:
        raise ValueError(f"{row.segment_id}: expected 16 kHz audio")
    if clean.shape != (48_000,) or noisy.shape != clean.shape:
        raise ValueError(f"{row.segment_id}: expected a 3-second mono pair")

    return clean, noisy

print(f"Validation: {len(validation):,} pairs")

Validation: 164 pairs


# 1. OM-LSA + MCRA
https://www.sciencedirect.com/science/article/abs/pii/S0165168401001281?

In [7]:
# OM-LSA + MCRA
def denoise_omlsa_mcra(noisy, *, gain_floor_db=-25, alpha_d=0.95, delta=5.0):
    n_fft = 512
    hop = 128
    block_frames = 125
    eps = 1e-12

    _, _, spectrum = stft(
        noisy, fs=SR, window="hann", nperseg=n_fft, noverlap=n_fft - hop
    )
    power = np.abs(spectrum) ** 2
    smoothed_power = convolve1d(power, [0.25, 0.50, 0.25], axis=0)

    # Initialize from the quietest 20% of frames in the first ~0.25 seconds.
    warmup = power[:, 1:33]
    quiet = np.argsort(warmup.mean(axis=0))[:max(1, warmup.shape[1] // 5)]
    noise_psd = np.maximum(warmup[:, quiet].mean(axis=1), eps)

    smoothed = smoothed_power[:, 0].copy()
    block_min = smoothed.copy()
    previous_block_min = np.full_like(smoothed, np.inf)
    tracking_probability = np.zeros_like(smoothed)
    previous_speech_snr = np.zeros_like(smoothed)
    log_floor = gain_floor_db * np.log(10) / 20
    enhanced_spectrum = np.empty_like(spectrum)

    for t, frame_power in enumerate(power.T):
        # MCRA speech-presence estimate for the noise update.
        smoothed = 0.8 * smoothed + 0.2 * smoothed_power[:, t]
        if t and t % block_frames == 0:
            previous_block_min = block_min
            block_min = smoothed.copy()
        else:
            block_min = np.minimum(block_min, smoothed)

        minimum = np.maximum(np.minimum(previous_block_min, block_min), eps)
        speech_detected = (smoothed > delta * minimum) | (
            frame_power > 1.8 * delta * minimum
        )
        tracking_probability = 0.2 * tracking_probability + 0.8 * speech_detected

        # OM-LSA gain using a fixed prior speech-absence probability of 0.5.
        gamma = np.clip(frame_power / noise_psd, eps, 1000.0)
        xi = 0.92 * previous_speech_snr + 0.08 * np.maximum(gamma - 1, 0)
        xi = np.maximum(xi, 10 ** (-25 / 10))
        ratio = xi / (1 + xi)
        v = gamma * ratio

        log_speech_gain = np.minimum(np.log(ratio) + 0.5 * exp1(v), 0.0)
        speech_probability = expit(v - np.log1p(xi))
        log_gain = speech_probability * log_speech_gain
        log_gain += (1 - speech_probability) * log_floor
        gain = np.exp(np.maximum(log_gain, log_floor))

        enhanced_spectrum[:, t] = gain * spectrum[:, t]
        previous_speech_snr = np.exp(2 * log_speech_gain) * gamma

        # Slow down the noise update when speech is likely present.
        smoothing = alpha_d + (1 - alpha_d) * tracking_probability
        noise_psd = np.maximum(
            smoothing * noise_psd + (1 - smoothing) * frame_power, eps
        )

    _, enhanced = istft(
        enhanced_spectrum, fs=SR, window="hann",
        nperseg=n_fft, noverlap=n_fft - hop,
    )
    return enhanced[:len(noisy)]

In [8]:
# Listen to One Validation Pair
row = validation.loc[validation["noise_category"].eq("alarm_siren")].iloc[0]
clean, noisy = read_validation_pair(row)
enhanced = denoise_omlsa_mcra(noisy)

print(f"{row.segment_id} | {row.noise_category} | {row.target_snr_db:g} dB")

for label, audio in (("Clean", clean), ("Noisy", noisy), ("Enhanced", enhanced)):
    print(label)
    display(Audio(np.clip(audio, -1, 1), rate=SR, normalize=False))

print(f"Noisy WB-PESQ:    {pesq(SR, clean, noisy, 'wb'):.3f}")
print(f"Enhanced WB-PESQ: {pesq(SR, clean, enhanced, 'wb'):.3f}")

speaker015_utt01_seg0003 | alarm_siren | 5 dB
Clean


Noisy


Enhanced


Noisy WB-PESQ:    1.074
Enhanced WB-PESQ: 1.100


In [9]:
# Evaluate All Validation Pairs
records = []

for row in validation.itertuples(index=False):
    clean, noisy = read_validation_pair(row)
    enhanced = denoise_omlsa_mcra(noisy)

    try:
        noisy_score = pesq(SR, clean, noisy, "wb")
        enhanced_score = pesq(SR, clean, enhanced, "wb")
    except Exception as error:
        raise RuntimeError(f"WB-PESQ failed for {row.segment_id}") from error

    records.append({
        "segment_id": row.segment_id,
        "noise_category": row.noise_category,
        "target_snr_db": row.target_snr_db,
        "Noisy PESQ": noisy_score,
        "Enhanced PESQ": enhanced_score,
        "PESQ Gain": enhanced_score - noisy_score,
    })

results = pd.DataFrame(records)

print(f"Evaluated: {len(results)}/{len(validation)} pairs")
display(results[["Noisy PESQ", "Enhanced PESQ", "PESQ Gain"]]
        .mean().to_frame().T.rename(index={0: "Overall"}).round(3))

def summarize_pesq(column):
    return results.groupby(column).agg(
        Pairs=("segment_id", "size"),
        Noisy_PESQ=("Noisy PESQ", "mean"),
        Enhanced_PESQ=("Enhanced PESQ", "mean"),
        PESQ_Gain=("PESQ Gain", "mean"),
    ).round(3)

display(summarize_pesq("target_snr_db"))
display(summarize_pesq("noise_category").sort_values("Pairs", ascending=False))

Evaluated: 164/164 pairs


,Noisy PESQ,Enhanced PESQ,PESQ Gain
Overall,1.395,1.577,0.183


,Pairs,Noisy_PESQ,Enhanced_PESQ,PESQ_Gain
target_snr_db,,,,
0.0,15,1.080,1.161,0.082
5.0,38,1.218,1.408,0.190
10.0,55,1.358,1.593,0.235
15.0,56,1.635,1.788,0.153


,Pairs,Noisy_PESQ,Enhanced_PESQ,PESQ_Gain
noise_category,,,,
machine,34,1.416,1.580,0.165
wind,26,1.474,1.669,0.196
alarm_siren,23,1.311,1.520,0.209
aircraft_cockpit,14,1.173,1.460,0.287
broadband_noise,10,1.115,1.413,0.298
traffic_vehicle,9,1.525,1.561,0.036
destroyer_engine,5,1.248,1.594,0.346
water,5,1.232,1.215,-0.017
military_vehicle,5,1.421,1.613,0.191


# 2. Generalized Subspace Filter
https://pubmed.ncbi.nlm.nih.gov/25474814/

In [10]:
from scipy.linalg import eigh
from numpy.lib.stride_tricks import sliding_window_view

def denoise_subspace(noisy, *, mu=1.5):
    order, stride, padding = 32, 8, 16
    frames = sliding_window_view(np.pad(noisy, (padding, padding)), order)[::stride]
    energy = np.mean(frames**2, axis=1)
    window = np.hanning(order + 2)[1:-1]
    identity = np.eye(order)

    output = np.zeros(len(noisy) + 2 * padding)
    weights = np.zeros_like(output)

    for start in range(0, len(frames), 250):
        end = min(start + 250, len(frames))
        lo = max(0, start - 500)
        hi = min(len(frames), end + 500)
        context = frames[lo:hi]

        # 주변 구간에서 에너지가 낮은 프레임으로 잡음 공분산 추정
        quiet_indices = np.argsort(energy[lo:hi])[:max(order, len(context) // 5)]
        quiet = context[quiet_indices]
        noise_cov = quiet.T @ quiet / len(quiet)
        noise_cov += (0.02 * np.trace(noise_cov) / order + 1e-10) * identity
        noisy_cov = context.T @ context / len(context)

        # 잡음 대비 에너지가 큰 성분은 남기고 작은 성분은 감쇠
        eigenvalues, basis = eigh(noisy_cov, noise_cov)
        speech_snr = np.maximum(eigenvalues - 1, 0)
        gain = np.maximum(speech_snr / (speech_snr + mu), 0.12)
        filtered = ((frames[start:end] @ basis) * gain) @ (basis.T @ noise_cov)

        # 겹치는 프레임을 합쳐 파형 복원
        positions = np.arange(start, end) * stride
        for offset, weight in enumerate(window):
            indices = positions + offset
            output[indices] += filtered[:, offset] * weight
            weights[indices] += weight

    output /= np.maximum(weights, 1e-12)
    return output[padding:padding + len(noisy)]

In [11]:
# 같은 알람 예시로 바로 비교
row = validation.loc[validation["noise_category"].eq("alarm_siren")].iloc[0]
clean, noisy = read_validation_pair(row)

for name, audio in [
    ("Noisy", noisy),
    ("OM-LSA + MCRA", denoise_omlsa_mcra(noisy)),
    ("Subspace", denoise_subspace(noisy)),
]:
    print(f"{name}: {pesq(SR, clean, audio, 'wb'):.3f}")
    display(Audio(audio, rate=SR, normalize=False))


Noisy: 1.074


OM-LSA + MCRA: 1.100


Subspace: 1.208


In [13]:
# 전체 164쌍과 잡음별 평균 비교
from tqdm.auto import tqdm

subspace_scores = {}

for row in tqdm(validation.itertuples(index=False), total=len(validation)):
    clean, noisy = read_validation_pair(row)
    enhanced = denoise_subspace(noisy)
    subspace_scores[row.segment_id] = pesq(SR, clean, enhanced, "wb")

comparison = results.rename(columns={"Enhanced PESQ": "OM-LSA PESQ"}).copy()
comparison["Subspace PESQ"] = comparison["segment_id"].map(subspace_scores)
assert comparison["Subspace PESQ"].notna().all()

columns = ["Noisy PESQ", "OM-LSA PESQ", "Subspace PESQ"]
print(f"평가 완료: {len(comparison)}쌍")
display(comparison[columns].mean().to_frame("전체 평균").T.round(3))
display(comparison.groupby("noise_category")[columns].agg(["mean", "size"]).round(3))

  0%|          | 0/164 [00:00<?, ?it/s]

평가 완료: 164쌍


,Noisy PESQ,OM-LSA PESQ,Subspace PESQ
전체 평균,1.395,1.577,1.486


Noisy PESQ      OM-LSA PESQ      Subspace PESQ     
                           mean size        mean size          mean size
noise_category                                                          
aircraft_cockpit          1.173   14       1.460   14         1.355   14
alarm_siren               1.311   23       1.520   23         1.433   23
animal                    1.423    2       1.327    2         1.461    2
bell                      1.675    3       1.430    3         1.768    3
broadband_noise           1.115   10       1.413   10         1.282   10
clap                      1.378    1       1.261    1         1.421    1
click                     2.129    1       1.860    1         2.282    1
destroyer_engine          1.248    5       1.594    5         1.401    5
destroyer_operations      1.286    5       1.526    5         1.448    5
door_moving               1.777    1       1.670    1         1.772    1
factory                   1.413    5       1.513    5         1.540    5
footsteps                 1.217    1       1.187    1         1.221    1
hf_channel                1.149    4       1.308    4         1.269    4
machine                   1.416   34       1.580   34         1.481   34
machine_gun               2.085    1       1.903    1         2.093    1
military_vehicle          1.421    5       1.613    5         1.466    5
phone_dialing             2.086    4       2.700    4         1.936    4
shower                    1.513    1       2.083    1         1.923    1
tooth_brushing            1.336    1       1.399    1         1.543    1
traffic_vehicle           1.525    9       1.561    9         1.612    9
vehicle_interior          1.861    3       2.109    3         1.849    3
water                     1.232    5       1.215    5         1.412    5
wind                      1.474   26       1.669   26         1.499   26

# 3. TSNR

In [12]:
import numpy as np
from scipy.signal import stft, istft
from scipy.ndimage import uniform_filter1d
from pesq import pesq
from tqdm.auto import tqdm


def denoise_tsnr(noisy, sr=16000):
    n_fft, hop = 512, 128
    smoothing, gain_floor = 0.98, 0.05

    _, times, spectrum = stft(
        noisy, fs=sr, window="hann",
        nperseg=n_fft, noverlap=n_fft - hop,
    )
    power = np.abs(spectrum) ** 2

    # 양끝의 패딩을 제외하고 에너지가 가장 낮은 20%로 잡음 파워 추정
    centers = np.rint(times * sr).astype(int)
    complete = np.flatnonzero(
        (centers >= n_fft // 2) & (centers + n_fft // 2 <= len(noisy))
    )
    energy = power[:, complete].mean(axis=0)
    count = int(np.ceil(0.20 * len(complete)))
    quiet = complete[np.argsort(energy, kind="stable")[:count]]
    noise_power = power[:, quiet].mean(axis=1)
    noise_power = np.maximum(
        uniform_filter1d(noise_power, size=3, mode="nearest"), 1e-18
    )

    # 1단계. 직전 프레임의 추정치를 이용해 1차 음성 파워 계산
    first_pass_power = np.empty_like(power)
    previous_power = np.zeros_like(noise_power)

    for frame, frame_power in enumerate(power.T):
        instantaneous_snr = np.maximum(frame_power / noise_power - 1, 0)
        prior_snr = instantaneous_snr
        if frame > 0:
            prior_snr = (
                smoothing * previous_power / noise_power
                + (1 - smoothing) * instantaneous_snr
            )

        gain = np.clip(prior_snr / (1 + prior_snr), gain_floor, 1)
        previous_power = np.abs(gain * spectrum[:, frame]) ** 2
        first_pass_power[:, frame] = previous_power

    # 2단계. 현재 프레임의 1차 추정치로 SNR을 다시 계산
    refined_snr = first_pass_power / noise_power[:, None]
    final_gain = np.clip(refined_snr / (1 + refined_snr), gain_floor, 1)

    _, enhanced = istft(
        final_gain * spectrum, fs=sr, window="hann",
        nperseg=n_fft, noverlap=n_fft - hop,
    )
    return enhanced[:len(noisy)]


tsnr_scores = []

for row in tqdm(validation.itertuples(index=False), total=len(validation)):
    clean, noisy = read_validation_pair(row)
    enhanced = denoise_tsnr(noisy, SR)
    tsnr_scores.append(pesq(SR, clean, enhanced, "wb"))

mean_pesq = float(np.mean(tsnr_scores))
print(f"TSNR 평균 WB-PESQ ({len(tsnr_scores)}쌍): {mean_pesq:.3f}")
print(f"기존 OM-LSA 1.577 대비: {mean_pesq - 1.577:+.3f}")
print(f"목표 2.300 대비: {mean_pesq - 2.300:+.3f}")

  0%|          | 0/164 [00:00<?, ?it/s]

TSNR 평균 WB-PESQ (164쌍): 1.549
기존 OM-LSA 1.577 대비: -0.028
목표 2.300 대비: -0.751


# 4. Dynamic EQ

In [13]:
import numpy as np
from scipy.ndimage import gaussian_filter, maximum_filter1d, uniform_filter1d
from scipy.signal import stft, istft


def denoise_dynamic_eq(noisy, sr=16000, *, threshold_db=10.0, max_cut_db=12.0):
    n_fft, hop = 1024, 128
    frequencies, _, spectrum = stft(
        noisy, fs=sr, window="hann", nperseg=n_fft, noverlap=n_fft - hop
    )
    magnitude_db = 20 * np.log10(np.maximum(np.abs(spectrum), 1e-12))

    # 양옆 주파수의 중앙값을 기준으로 피크가 얼마나 튀는지 계산
    padded = np.pad(magnitude_db, ((10, 10), (0, 0)), mode="edge")
    sidebands = np.stack([
        padded[10 + offset:10 + offset + len(frequencies)]
        for offset in (*range(-10, -3), *range(4, 11))
    ])
    prominence_db = magnitude_db - np.median(sidebands, axis=0)

    # 알람 후보 대역에서 좁고 지속적인 피크 검출
    target = (
        ((frequencies >= 850) & (frequencies <= 1050))
        | ((frequencies >= 1400) & (frequencies <= 1620))
    )
    local_peak = magnitude_db == maximum_filter1d(magnitude_db, size=5, axis=0)
    candidates = target[:, None] & local_peak & (prominence_db >= threshold_db)

    # 인접 주파수로 조금 이동해도 같은 피크로 보고, 7프레임 중 4개 이상 확인
    nearby = maximum_filter1d(candidates.astype(float), size=5, axis=0)
    persistent = uniform_filter1d(nearby, size=7, axis=1, mode="constant") >= 0.55

    # 피크 주변까지 감쇠하고, 시간·주파수 방향으로 경계를 부드럽게 처리
    cut_db = np.where(candidates & persistent, prominence_db - threshold_db, 0.0)
    cut_db = maximum_filter1d(np.minimum(cut_db, max_cut_db), size=5, axis=0)
    cut_db = gaussian_filter(cut_db, sigma=(0.8, 1.0))
    gain = 10 ** (-cut_db / 20)

    _, enhanced = istft(
        spectrum * gain, fs=sr, window="hann", nperseg=n_fft, noverlap=n_fft - hop
    )
    return enhanced[:len(noisy)]

In [14]:
import pandas as pd
from IPython.display import display
from pesq import pesq
from tqdm.auto import tqdm

scores = []
for row in tqdm(validation.itertuples(index=False), total=len(validation)):
    clean, noisy = read_validation_pair(row)
    enhanced = denoise_dynamic_eq(noisy, sr=SR)

    scores.append({
        "noise_category": row.noise_category,
        "Noisy PESQ": pesq(SR, clean, noisy, "wb"),
        "Dynamic EQ PESQ": pesq(SR, clean, enhanced, "wb"),
    })

eq_results = pd.DataFrame(scores)
eq_results["PESQ 변화"] = eq_results["Dynamic EQ PESQ"] - eq_results["Noisy PESQ"]

metrics = ["Noisy PESQ", "Dynamic EQ PESQ", "PESQ 변화"]
alarm = eq_results.loc[eq_results["noise_category"].eq("alarm_siren")]

display(pd.DataFrame({
    f"전체 ({len(eq_results)}쌍)": eq_results[metrics].mean(),
    f"알람·사이렌 ({len(alarm)}쌍)": alarm[metrics].mean(),
}).T.round(3))

  0%|          | 0/164 [00:00<?, ?it/s]

,Noisy PESQ,Dynamic EQ PESQ,PESQ 변화
전체 (164쌍),1.395,1.383,-0.011
알람·사이렌 (23쌍),1.311,1.318,0.008
